# 🏭 Комплексное тестирование модуля построения модели (3-Statement Model)

**Полная переработанная версия** с учетом всех модулей: макроанализ, стресс-тест, рейтингование, ковенанты.

## 📋 Структура ноутбука:

1. **Подготовка окружения и загрузка конфигурации** - ROOT, COMPANY, project.yaml, режим модели
2. **Загрузка и валидация исторических данных** - IS, BS, CF, проверка полноты
3. **Анализ макро-факторов и регрессионный анализ драйверов** - Revenue regression, COGS, SG&A
4. **Построение модели** - запуск build_model для выбранного режима
5. **Валидация результатов модели** - BS Identity, CF Identity, Retained Earnings
6. **Анализ корков (Corkscrews)** - PP&E, Debt, WC, Tax, Equity
7. **Визуальный анализ результатов** - графики финансовых отчетов и ключевых метрик
8. **Тестирование для разных режимов модели** - Lite, Standard, Custom
9. **Стресс-тестирование** - Base, Bear, Bull сценарии
10. **Рейтингование** - Base, Forecast, Stress рейтинги
11. **Регрессионный анализ драйверов с макропараметрами (детальный)** - Revenue regression summary, train/test анализ
12. **Тестирование ковенант (Covenants)** - Net Debt/EBITDA, ICR, DSCR, Debt/Equity, FFO/NetDebt, Debt/FFO
13. **Комплексное тестирование данных** - TestSuite, полнота, консистентность
14. **Итоговая сводка и отчетность** - сводка всех проверок

---

**Примечание**: Этот ноутбук использует модули:
- `engine.model.core.build_model()` - построение модели
- `engine.acceptance.checks.run_acceptance()` - валидация
- `engine.acceptance.covenants.run_covenants()` - расчет ковенант
- `engine.stress_rating.core.run_stress()` - стресс-тест
- `engine.stress_rating.core.run_rating()` - рейтинг
- `tools.test_suite.TestSuite` - комплексное тестирование


In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY: COMPANY = 'test_metals'  # fallback
print(f'Company: {COMPANY}, ROOT: {ROOT}')


## 🔧 1️⃣ Подготовка окружения и загрузка конфигурации <a id="section-1"></a>


In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Dict, List, Any

# Настройка для графиков в Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
sns.set_palette("husl")

# Определение корня проекта
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

# Добавляем корень проекта в sys.path ПЕРЕД импортом engine
sys.path.insert(0, str(ROOT))

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = COMPANY  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

# Импорт engine модулей ПОСЛЕ настройки sys.path
from engine.database.data_mart import get_data_mart

# Примечание о новой архитектуре:
# Витрина данных (FinancialDataMart) использует модульную архитектуру:
# - Внутри используются repositories (history_repository, canonical_repository, macro_repository и т.д.)
# - Внутри используются services (normalization_service, validation_service)
# - Внешний API остался прежним для обратной совместимости
# - Можно использовать repositories напрямую: mart.macro_repository, mart.history_repository и т.д.
from engine.model.core import build_model
from engine.acceptance.checks import run_acceptance
from engine.acceptance.covenants import run_covenants
from engine.stress_rating.core import run_stress, run_rating
from tools.test_suite import TestSuite

MODEL_VERSION: Optional[str] = None


### 1.1 Загрузка конфигурации из project.yaml


In [ ]:
display(Markdown("### Конфигурация из project.yaml"))

proj_yaml_path = croot / "configs" / "project.yaml"
if not proj_yaml_path.exists():
    print(f"❌ Файл конфигурации не найден: {proj_yaml_path}")
    proj_yaml = {}
else:
    proj_yaml = yaml.safe_load(proj_yaml_path.read_text(encoding='utf-8'))
    
    print(f"✅ Конфигурация загружена")
    
    # Режим модели
    model_mode = proj_yaml.get('model', {}).get('mode', 'standard')
    print(f"  - Режим модели: {model_mode}")
    
    # Параметры периодов
    periods_config = proj_yaml.get("model", {}).get("standard", {}).get("periods", {})
    HISTORY_START_YEAR = periods_config.get("history_start_year", 2010)
    HISTORY_END_YEAR = periods_config.get("history_end_year", 2024)
    FORECAST_START_YEAR = periods_config.get("forecast_start_year", 2025)
    FORECAST_END_YEAR = periods_config.get("forecast_end_year", 2029)
    
    print(f"  - История: {HISTORY_START_YEAR}-{HISTORY_END_YEAR}")
    print(f"  - Прогноз: {FORECAST_START_YEAR}-{FORECAST_END_YEAR}")
    
    # Revenue настройки
    if 'model' in proj_yaml and 'standard' in proj_yaml['model']:
        std_cfg = proj_yaml['model']['standard']
        rev_cfg = std_cfg.get('revenue', {})
        print(f"  - Revenue driver: {rev_cfg.get('driver', 'N/A')}")
        print(f"  - Revenue method: {'elastic_net' if rev_cfg.get('use_elastic_net', False) else rev_cfg.get('fallback_growth', 'N/A')}")
    
    # Ковенанты
    covenant_cfg = proj_yaml.get('covenants', {})
    if covenant_cfg and covenant_cfg.get('enabled', False):
        thresholds = covenant_cfg.get('thresholds', {})
        print(f"  - Ковенанты включены")
        print(f"    Net Debt/EBITDA max: {thresholds.get('net_debt_to_ebitda_max', 'N/A')}")
        print(f"    ICR min: {thresholds.get('interest_coverage_min', 'N/A')}")


### 1.2 Инициализация DataMart


In [ ]:
# Вспомогательные функции для работы с DataMart
def _open_data_mart():
    try:
        return get_data_mart(ROOT, COMPANY)
    except Exception as exc:
        print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
        return None

def _resolve_model_version(mart):
    global MODEL_VERSION
    if MODEL_VERSION:
        return MODEL_VERSION
    try:
        versions = mart.get_existing_versions()
        if versions:
            MODEL_VERSION = versions[0]['version']
            return MODEL_VERSION
    except Exception as exc:
        print(f"⚠️ Не удалось определить версию модели: {exc}")
    return None

# Проверка подключения
mart = _open_data_mart()
if mart is not None:
    print("✅ DataMart подключен")
    versions = mart.get_existing_versions()
    if versions:
        print(f"📦 Доступные версии модели: {[v['version'] for v in versions[:5]]}")
    mart.close()
else:
    print("❌ Не удалось подключиться к DataMart")


## 2️⃣ Загрузка и валидация исторических данных <a id="section-2"></a>


### 2.0 Обновление corkscrew таблиц и sanity-check

Перед любыми проверками истории выполняем миграцию корксрудов и быстрый sanity-check. Эти команды используют текущую `COMPANY` и корень проекта.


In [ ]:
import subprocess

display(Markdown("#### 2.0.1 Миграция корксрудов из history"))

hist_years = []
mart = _open_data_mart()
if mart is not None:
    hist_bs = mart.db.load_history_bs(COMPANY)
    mart.close()
    if not hist_bs.empty:
        hist_years = sorted(int(col) for col in hist_bs.columns if str(col).isdigit())
if not hist_years:
    hist_years = list(range(2010, 2025))

migration_cmd = [
    sys.executable,
    str(ROOT / 'tools' / 'migrations' / 'load_historical_data_to_corkscrews.py'),
    '--company', COMPANY,
    '--years',
    *[str(y) for y in hist_years],
]
print('▶️ Running:', ' '.join(migration_cmd))
result = subprocess.run(migration_cmd, cwd=ROOT)
if result.returncode != 0:
    raise RuntimeError('load_historical_data_to_corkscrews.py завершился с ошибкой')
print('✅ Corkscrew таблицы обновлены')


In [ ]:
import subprocess

display(Markdown("#### 2.0.2 Быстрая проверка validate_corkscrew_history.py"))
validate_cmd = [
    sys.executable,
    str(ROOT / 'tools' / 'validate_corkscrew_history.py'),
    '--company', COMPANY,
    '--tolerance', '1000000',
]
print('▶️ Running:', ' '.join(validate_cmd))
result = subprocess.run(validate_cmd, cwd=ROOT)
if result.returncode != 0:
    raise RuntimeError('validate_corkscrew_history.py обнаружил расхождения')
print('✅ Sanity-check завершился без расхождений')


In [ ]:
# Функции загрузки данных
def load_history_from_db(statement_type='IS', canonical=False):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        try:
            if stmt == 'IS':
                df = mart.get_history_income_statement(canonical=canonical)
            elif stmt == 'BS':
                df = mart.get_history_balance_sheet(canonical=canonical)
            else:
                df = mart.get_history_cash_flow(canonical=canonical)
            if df is not None and not df.empty:
                print(f"✅ История {stmt} загружена из витрины")
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить историю {stmt} из витрины: {exc}")
        finally:
            mart.close()
    return df if not df.empty else pd.DataFrame()

def load_model_forecast_from_db(statement_type='IS', canonical=True):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        version = _resolve_model_version(mart)
        if version:
            try:
                df = mart.get_model_results(version, stmt, canonical=canonical)
                if df is not None and not df.empty:
                    print(f"✅ Прогноз {stmt} загружен из витрины (версия {version})")
            except Exception as exc:
                print(f"⚠️ Не удалось загрузить прогноз {stmt}: {exc}")
            finally:
                mart.close()
    return df if not df.empty else pd.DataFrame()

# Загружаем исторические данные
display(Markdown("### Загрузка исторических данных"))

hist_is = load_history_from_db('IS', canonical=True)
hist_bs = load_history_from_db('BS', canonical=True)
hist_cf = load_history_from_db('CF', canonical=True)

print(f"\n📊 Загружено:")
print(f"  - IS: {len(hist_is)} строк" if not hist_is.empty else "  - IS: нет данных")
print(f"  - BS: {len(hist_bs)} строк" if not hist_bs.empty else "  - BS: нет данных")
print(f"  - CF: {len(hist_cf)} строк" if not hist_cf.empty else "  - CF: нет данных")


### 2.1 Проверка полноты данных для выбранного режима


In [ ]:
# Проверка доступности метрик
mart = _open_data_mart()
if mart is not None:
    detected_model_type = mart.detect_model_type()
    config_model_mode = proj_yaml.get('model', {}).get('mode', 'standard')
    
    print(f"📋 Тип модели в конфиге: {config_model_mode}")
    print(f"🔍 Определенный тип модели из истории: {detected_model_type}")
    
    if detected_model_type != config_model_mode:
        print(f"⚠️ ВНИМАНИЕ: Тип модели в конфиге ({config_model_mode}) не совпадает с определенным из истории ({detected_model_type})")
    
    mart.close()
else:
    print("❌ Не удалось подключиться к DataMart")


## 3️⃣ Анализ макро-факторов и регрессионный анализ драйверов <a id="section-3"></a>


### 3.1 Загрузка макро-прогнозов


In [ ]:
display(Markdown("### Макро-факторы для регрессии"))

macro_cfg = proj_yaml.get("macro_forecast", {})
factors = macro_cfg.get("factors", [])

mart = _open_data_mart()
macro_data = {}
if mart is not None:
    for factor in factors:
        try:
            forecast = mart.get_macro_forecast(factor) or {}
            history = mart.get_macro_history(factor) or {}
            macro_data[factor] = {
                'forecast_years': len(forecast),
                'history_years': len(history),
                'forecast_start': min(forecast) if forecast else None,
                'forecast_end': max(forecast) if forecast else None
            }
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить {factor}: {exc}")
    mart.close()

if macro_data:
    macro_df = pd.DataFrame(macro_data).T
    display(macro_df)
else:
    print("⚠️ Макро-факторы не найдены")


### 3.2 Регрессионный анализ Revenue с макропараметрами


In [ ]:
display(Markdown("### Анализ Revenue регрессии"))

# Загружаем результаты регрессии если они есть
regression_summary_path = croot / "outputs" / "model" / "revenue_regression_summary.csv"
train_fitted_path = croot / "outputs" / "model" / "train_period_fitted.csv"
test_forecasts_path = croot / "outputs" / "model" / "test_period_forecasts.csv"

if regression_summary_path.exists():
    regression_summary = pd.read_csv(regression_summary_path)
    display(Markdown("#### Revenue Regression Summary"))
    display(regression_summary)
    
    # Визуализация коэффициентов
    if 'coefficient' in regression_summary.columns and 'feature' in regression_summary.columns:
        fig, ax = plt.subplots(figsize=(10, 6))
        coef_df = regression_summary[['feature', 'coefficient']].sort_values('coefficient', key=abs, ascending=False)
        ax.barh(coef_df['feature'], coef_df['coefficient'])
        ax.set_xlabel('Coefficient')
        ax.set_title('Revenue Regression Coefficients')
        ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Файл revenue_regression_summary.csv не найден. Запустите build_model для генерации.")

# Train/Test анализ
if train_fitted_path.exists() and test_forecasts_path.exists():
    train_fitted = pd.read_csv(train_fitted_path)
    test_forecasts = pd.read_csv(test_forecasts_path)
    
    display(Markdown("#### Train/Test Analysis"))
    
    # График Actual vs Fitted/Forecast
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if 'year' in train_fitted.columns and 'actual' in train_fitted.columns and 'fitted' in train_fitted.columns:
        ax.plot(train_fitted['year'], train_fitted['actual'], 'o-', label='Actual (Train)', color='blue')
        ax.plot(train_fitted['year'], train_fitted['fitted'], 's-', label='Fitted (Train)', color='green')
    
    if 'year' in test_forecasts.columns and 'actual' in test_forecasts.columns and 'forecast' in test_forecasts.columns:
        ax.plot(test_forecasts['year'], test_forecasts['actual'], 'o-', label='Actual (Test)', color='orange')
        ax.plot(test_forecasts['year'], test_forecasts['forecast'], 's-', label='Forecast (Test)', color='red')
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Revenue')
    ax.set_title('Revenue: Actual vs Fitted/Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Метрики качества
    if 'residual' in train_fitted.columns:
        train_mape = np.mean(np.abs(train_fitted['residual'] / train_fitted['actual'])) * 100 if 'actual' in train_fitted.columns else None
        print(f"Train MAPE: {train_mape:.2f}%" if train_mape else "Train MAPE: N/A")
    
    if 'residual' in test_forecasts.columns:
        test_mape = np.mean(np.abs(test_forecasts['residual'] / test_forecasts['actual'])) * 100 if 'actual' in test_forecasts.columns else None
        print(f"Test MAPE: {test_mape:.2f}%" if test_mape else "Test MAPE: N/A")


## 4️⃣ Построение модели <a id="section-4"></a>


In [ ]:
display(Markdown("### Запуск построения модели"))

print(f"Запуск построения модели для {COMPANY}...")
print(f"Режим модели: {model_mode}")

try:
    MODEL_VERSION = build_model(ROOT, COMPANY)
    print(f"\n✅ Модель построена успешно!")
    print(f"📦 Версия модели: {MODEL_VERSION}")
    
    # Проверяем результаты
    mart = _open_data_mart()
    if mart is not None:
        is_forecast = mart.get_model_results(MODEL_VERSION, 'IS', canonical=True)
        if not is_forecast.empty:
            forecast_years = [int(c) for c in is_forecast.columns if str(c).isdigit()]
            forecast_years = sorted(forecast_years)
            if forecast_years:
                print(f"\n📊 Годы прогноза: {forecast_years[0]}-{forecast_years[-1]} ({len(forecast_years)} лет)")
        mart.close()
except Exception as e:
    print(f"\n❌ Ошибка построения модели: {e}")
    import traceback
    traceback.print_exc()
    raise


## 5️⃣ Валидация результатов модели <a id="section-5"></a>


### 5.1 Balance Sheet Identity


In [ ]:
display(Markdown("### Проверка BS Identity: Assets = Liabilities + Equity"))

mart = _open_data_mart()
if mart is not None:
    bs_df = mart.get_model_results(MODEL_VERSION, 'BS', canonical=True)
    
    if not bs_df.empty:
        # Преобразуем в wide format
        bs_wide = bs_df.set_index('metric')
        year_cols = [c for c in bs_wide.columns if str(c).isdigit()]
        
        bs_checks = []
        for year in sorted([int(c) for c in year_cols]):
            year_str = str(year)
            if year_str in bs_wide.columns:
                total_assets = bs_wide.loc['total_assets', year_str] if 'total_assets' in bs_wide.index else 0.0
                total_liabilities = bs_wide.loc['total_liabilities', year_str] if 'total_liabilities' in bs_wide.index else 0.0
                total_equity = bs_wide.loc['total_equity', year_str] if 'total_equity' in bs_wide.index else 0.0
                
                total_le = total_liabilities + total_equity
                diff = total_assets - total_le
                
                bs_checks.append({
                    'Year': year,
                    'Assets': f"{total_assets:,.0f}",
                    'Liabilities': f"{total_liabilities:,.0f}",
                    'Equity': f"{total_equity:,.0f}",
                    'Liab+Equity': f"{total_le:,.0f}",
                    'Difference': f"{diff:,.0f}",
                    'Status': '✅' if abs(diff) < 1.0 else '❌'
                })
        
        bs_check_df = pd.DataFrame(bs_checks)
        display(bs_check_df)
        
        # Визуализация
        if bs_checks:
            fig, ax = plt.subplots(figsize=(10, 6))
            years = [c['Year'] for c in bs_checks]
            diffs = [float(c['Difference'].replace(',', '')) for c in bs_checks]
            ax.plot(years, diffs, 'o-', color='red' if any(abs(d) > 1.0 for d in diffs) else 'green')
            ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
            ax.set_xlabel('Year')
            ax.set_ylabel('Difference (Assets - Liab+Equity)')
            ax.set_title('BS Identity Check')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    
    mart.close()


### 5.2 Cash Flow Identity


In [ ]:
display(Markdown("### Проверка CF Identity: Net Change = CFO + CFI + CFF + FX Effect"))

mart = _open_data_mart()
if mart is not None:
    cf_df = mart.get_model_results(MODEL_VERSION, 'CF', canonical=True)
    
    if not cf_df.empty:
        cf_wide = cf_df.set_index('metric')
        year_cols = [c for c in cf_wide.columns if str(c).isdigit()]
        
        cf_checks = []
        for year in sorted([int(c) for c in year_cols]):
            year_str = str(year)
            if year_str in cf_wide.columns:
                net_change = cf_wide.loc['net_change_in_cash', year_str] if 'net_change_in_cash' in cf_wide.index else 0.0
                cfo = cf_wide.loc['cfo_total', year_str] if 'cfo_total' in cf_wide.index else 0.0
                cfi = cf_wide.loc['cfi_total', year_str] if 'cfi_total' in cf_wide.index else 0.0
                cff = cf_wide.loc['cff_total', year_str] if 'cff_total' in cf_wide.index else 0.0
                fx_effect = cf_wide.loc.get('effect_of_exchange_rate_changes_on_cash', year_str, 0.0) if 'effect_of_exchange_rate_changes_on_cash' in cf_wide.index else 0.0
                
                total_cf = cfo + cfi + cff + fx_effect
                diff = net_change - total_cf
                
                cf_checks.append({
                    'Year': year,
                    'Net Change': f"{net_change:,.0f}",
                    'CFO': f"{cfo:,.0f}",
                    'CFI': f"{cfi:,.0f}",
                    'CFF': f"{cff:,.0f}",
                    'FX Effect': f"{fx_effect:,.0f}",
                    'Sum (CFO+CFI+CFF+FX)': f"{total_cf:,.0f}",
                    'Difference': f"{diff:,.0f}",
                    'Status': '✅' if abs(diff) < 1.0 else '❌'
                })
        
        cf_check_df = pd.DataFrame(cf_checks)
        display(cf_check_df)
    
    mart.close()


### 5.3 Acceptance Checks


In [ ]:
display(Markdown("### Запуск Acceptance Checks"))

try:
    run_acceptance(croot)
    print("✅ Acceptance checks выполнены")
    
    # Загружаем результаты
    mart = _open_data_mart()
    if mart is not None:
        acceptance_df = mart.get_output('acceptance_checks')
        if not acceptance_df.empty:
            display(Markdown("#### Результаты Acceptance Checks"))
            display(acceptance_df.head(20))
        mart.close()
except Exception as e:
    print(f"⚠️ Ошибка выполнения acceptance checks: {e}")


## 6️⃣ Анализ корков (Corkscrews) <a id="section-6"></a>


In [ ]:
display(Markdown("### Анализ корков"))

# Загружаем данные корков из DataMart
mart = _open_data_mart()
if mart is not None:
    # PP&E Corkscrew
    ppe_df = mart.get_output('ppe_schedule')
    if not ppe_df.empty:
        display(Markdown("#### PP&E Corkscrew"))
        display(ppe_df.head(10))
    
    # Debt Corkscrew
    debt_df = mart.get_output('debt_schedule')
    if not debt_df.empty:
        display(Markdown("#### Debt Corkscrew"))
        display(debt_df.head(10))
    
    # Working Capital
    wc_df = mart.get_output('working_capital')
    if not wc_df.empty:
        display(Markdown("#### Working Capital"))
        display(wc_df.head(10))
    
    mart.close()
else:
    print("❌ Не удалось подключиться к DataMart")


## 7️⃣ Визуальный анализ результатов <a id="section-7"></a>


In [ ]:
display(Markdown("### Графики финансовых отчетов"))

# Загружаем прогнозные данные
forecast_is = load_model_forecast_from_db('IS', canonical=True)
forecast_bs = load_model_forecast_from_db('BS', canonical=True)
forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Ключевые метрики для визуализации
key_metrics = {
    'IS': ['revenue', 'cogs', 'sga', 'ebit', 'net_income'],
    'BS': ['total_assets', 'total_liabilities', 'total_equity'],
    'CF': ['cfo_total', 'cfi_total', 'cff_total', 'net_change_in_cash']
}

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for idx, (stmt, metrics) in enumerate(key_metrics.items()):
    ax = axes[idx]
    
    # Объединяем историю и прогноз
    if stmt == 'IS':
        hist_df = hist_is
        forecast_df = forecast_is
    elif stmt == 'BS':
        hist_df = hist_bs
        forecast_df = forecast_bs
    else:
        hist_df = hist_cf
        forecast_df = forecast_cf
    
    # Строим графики для каждой метрики
    for metric in metrics:
        if not hist_df.empty and 'metric' in hist_df.columns:
            hist_metric = hist_df[hist_df['metric'] == metric]
            if not hist_metric.empty:
                year_cols = [c for c in hist_metric.columns if str(c).isdigit()]
                years = sorted([int(c) for c in year_cols])
                values = [hist_metric[str(y)].iloc[0] if str(y) in hist_metric.columns else np.nan for y in years]
                ax.plot(years, values, 'o-', label=f'{metric} (Hist)', alpha=0.7)
        
        if not forecast_df.empty and 'metric' in forecast_df.columns:
            forecast_metric = forecast_df[forecast_df['metric'] == metric]
            if not forecast_metric.empty:
                year_cols = [c for c in forecast_metric.columns if str(c).isdigit()]
                years = sorted([int(c) for c in year_cols])
                values = [forecast_metric[str(y)].iloc[0] if str(y) in forecast_metric.columns else np.nan for y in years]
                ax.plot(years, values, 's--', label=f'{metric} (Forecast)', alpha=0.7)
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Value')
    ax.set_title(f'{stmt} - Key Metrics')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.axvline(x=HISTORY_END_YEAR, color='red', linestyle=':', linewidth=1, label='History End')

plt.tight_layout()
plt.show()


## 8️⃣ Тестирование для разных режимов модели <a id="section-8"></a>


In [ ]:
display(Markdown("### Тестирование для разных режимов"))

# Проверяем текущий режим
print(f"Текущий режим модели: {model_mode}")

# Для тестирования других режимов нужно изменить project.yaml и перезапустить модель
print("\n📝 Для тестирования других режимов:")
print("  1. Измените project.yaml: model.mode = 'lite' | 'standard' | 'custom'")
print("  2. Перезапустите раздел 'Построение модели'")
print("  3. Проверьте результаты в разделах валидации")

# Проверяем доступные режимы
mart = _open_data_mart()
if mart is not None:
    detected_type = mart.detect_model_type()
    print(f"\n🔍 Определенный тип модели из данных: {detected_type}")
    mart.close()


## 9️⃣ Стресс-тестирование <a id="section-9"></a>


In [ ]:
display(Markdown("### Запуск стресс-тестирования"))

# Загружаем сценарии
scenarios_dir = croot / "configs" / "scenarios"
scenarios = []
if scenarios_dir.exists():
    for scenario_file in scenarios_dir.glob("*.yaml"):
        scenarios.append(scenario_file.stem)

if scenarios:
    print(f"📋 Найдено сценариев: {scenarios}")
    
    # Запускаем стресс-тест для каждого сценария
    stress_results = {}
    for scenario in scenarios:
        try:
            print(f"\n🔄 Запуск сценария: {scenario}")
            result = run_stress(ROOT, COMPANY, scenario_name=scenario, base_version=MODEL_VERSION)
            if result:
                stress_results[scenario] = result
                print(f"✅ Сценарий {scenario} выполнен")
        except Exception as e:
            print(f"⚠️ Ошибка выполнения сценария {scenario}: {e}")
    
    # Визуализация результатов
    if stress_results:
        display(Markdown("#### Сравнение сценариев"))
        
        # Собираем данные для сравнения
        comparison_data = []
        for scenario, result in stress_results.items():
            # Загружаем результаты из DataMart
            mart = _open_data_mart()
            if mart is not None:
                stress_is = mart.get_stress_results(scenario, 'IS')
                if not stress_is.empty:
                    # Извлекаем ключевые метрики
                    for year in stress_is.columns:
                        if str(year).isdigit():
                            revenue = stress_is.loc[stress_is['metric'] == 'revenue', str(year)].iloc[0] if 'revenue' in stress_is['metric'].values else None
                            ebit = stress_is.loc[stress_is['metric'] == 'ebit', str(year)].iloc[0] if 'ebit' in stress_is['metric'].values else None
                            if revenue is not None:
                                comparison_data.append({
                                    'Scenario': scenario,
                                    'Year': int(year),
                                    'Revenue': revenue,
                                    'EBIT': ebit
                                })
                mart.close()
        
        if comparison_data:
            comparison_df = pd.DataFrame(comparison_data)
            
            # График Revenue по сценариям
            fig, ax = plt.subplots(figsize=(12, 6))
            for scenario in comparison_df['Scenario'].unique():
                scenario_data = comparison_df[comparison_df['Scenario'] == scenario]
                ax.plot(scenario_data['Year'], scenario_data['Revenue'], 'o-', label=scenario)
            
            ax.set_xlabel('Year')
            ax.set_ylabel('Revenue')
            ax.set_title('Revenue by Stress Scenario')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
else:
    print("⚠️ Сценарии стресс-тестирования не найдены")
    print(f"   Ожидаемый путь: {scenarios_dir}")


## 🔟 Рейтингование <a id="section-10"></a>


In [ ]:
display(Markdown("### Расчет рейтингов"))

# Запускаем рейтингование
try:
    # Base rating (из истории)
    base_rating = run_rating(ROOT, COMPANY, rating_type='base', version=MODEL_VERSION)
    
    # Forecast rating (из прогноза)
    forecast_rating = run_rating(ROOT, COMPANY, rating_type='forecast', version=MODEL_VERSION)
    
    # Stress rating (из стресс-сценариев, если есть)
    stress_rating = None
    if scenarios:
        stress_rating = run_rating(ROOT, COMPANY, rating_type='stress', version=MODEL_VERSION)
    
    # Загружаем результаты
    mart = _open_data_mart()
    if mart is not None:
        ratings_df = mart.get_output('ratings')
        if not ratings_df.empty:
            display(Markdown("#### Результаты рейтингования"))
            display(ratings_df)
            
            # Визуализация рейтингов
            if 'year' in ratings_df.columns and 'rating_score' in ratings_df.columns:
                fig, ax = plt.subplots(figsize=(12, 6))
                for rating_type in ratings_df['rating_type'].unique() if 'rating_type' in ratings_df.columns else ['base']:
                    type_data = ratings_df[ratings_df['rating_type'] == rating_type] if 'rating_type' in ratings_df.columns else ratings_df
                    ax.plot(type_data['year'], type_data['rating_score'], 'o-', label=rating_type)
                
                ax.set_xlabel('Year')
                ax.set_ylabel('Rating Score')
                ax.set_title('Rating Scores Over Time')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
        mart.close()
    
    print("✅ Рейтингование выполнено")
    
except Exception as e:
    print(f"⚠️ Ошибка рейтингования: {e}")
    import traceback
    traceback.print_exc()


## 1️⃣1️⃣ Регрессионный анализ драйверов с макропараметрами (детальный) <a id="section-11"></a>


In [ ]:
display(Markdown("### Детальный анализ Revenue регрессии"))

# Этот раздел расширяет раздел 3.2 более детальным анализом
# Загружаем полный отчет регрессии
if regression_summary_path.exists():
    regression_summary = pd.read_csv(regression_summary_path)
    
    # Статистика модели
    if 'r_squared' in regression_summary.columns:
        r2 = regression_summary['r_squared'].iloc[0] if 'r_squared' in regression_summary.columns else None
        print(f"R²: {r2:.4f}" if r2 else "R²: N/A")
    
    if 'cv_rmse' in regression_summary.columns:
        cv_rmse = regression_summary['cv_rmse'].iloc[0] if 'cv_rmse' in regression_summary.columns else None
        print(f"CV RMSE: {cv_rmse:.4f}" if cv_rmse else "CV RMSE: N/A")
    
    # Feature importance (стандартизованные коэффициенты)
    if 'coefficient' in regression_summary.columns and 'feature' in regression_summary.columns:
        display(Markdown("#### Feature Importance"))
        importance_df = regression_summary[['feature', 'coefficient']].copy()
        importance_df['abs_coefficient'] = importance_df['coefficient'].abs()
        importance_df = importance_df.sort_values('abs_coefficient', ascending=False)
        display(importance_df)
        
        # График важности признаков
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(importance_df['feature'], importance_df['abs_coefficient'])
        ax.set_xlabel('Absolute Coefficient')
        ax.set_title('Feature Importance (Absolute Coefficients)')
        plt.tight_layout()
        plt.show()
    
    # Residuals analysis
    if train_fitted_path.exists():
        train_fitted = pd.read_csv(train_fitted_path)
        if 'residual' in train_fitted.columns:
            display(Markdown("#### Residuals Analysis"))
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            # Residuals vs Fitted
            axes[0].scatter(train_fitted['fitted'], train_fitted['residual'], alpha=0.6)
            axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
            axes[0].set_xlabel('Fitted Values')
            axes[0].set_ylabel('Residuals')
            axes[0].set_title('Residuals vs Fitted')
            axes[0].grid(True, alpha=0.3)
            
            # Residuals distribution
            axes[1].hist(train_fitted['residual'], bins=20, edgecolor='black', alpha=0.7)
            axes[1].set_xlabel('Residuals')
            axes[1].set_ylabel('Frequency')
            axes[1].set_title('Residuals Distribution')
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
else:
    print("⚠️ Файлы регрессии не найдены. Запустите build_model для генерации.")


## 1️⃣3️⃣ Комплексное тестирование данных <a id="section-13"></a>


In [ ]:
display(Markdown("### Запуск комплексного тестирования"))

# Определяем годы для тестирования
test_years = list(range(HISTORY_START_YEAR, FORECAST_END_YEAR + 1))

# Запускаем TestSuite
test_suite = TestSuite(COMPANY, ROOT)
test_suite.run_all_tests()

# Отображаем результаты
display(Markdown("#### Сводка результатов тестирования"))
print(f"\n📊 Итоги:")
print(f"  - Всего тестов: {test_suite.total_tests}")
print(f"  - Пройдено: {test_suite.passed}")
print(f"  - Провалено: {test_suite.failed}")
print(f"  - Предупреждений: {test_suite.warnings}")

# Детальные результаты
for result in test_suite.results:
    status_icon = '✅' if result['status'] == 'passed' else '❌' if result['status'] == 'failed' else '⚠️'
    print(f"\n{status_icon} {result['name']}: {result['status'].upper()}")
    print(f"   {result['message']}")

# Сохраняем отчет
output_dir = ROOT / "outputs" / "test_reports"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / f"{COMPANY}_full_test_report.json"
test_suite.generate_report(report_path)
print(f"\n📄 Отчет сохранен: {report_path}")


## 1️⃣4️⃣ Итоговая сводка и отчетность <a id="section-14"></a>


In [ ]:
display(Markdown("### Итоговая сводка всех проверок"))

summary = {
    'Компания': COMPANY,
    'Версия модели': MODEL_VERSION or 'N/A',
    'Режим модели': model_mode,
    'Период истории': f"{HISTORY_START_YEAR}-{HISTORY_END_YEAR}",
    'Период прогноза': f"{FORECAST_START_YEAR}-{FORECAST_END_YEAR}",
}

# Добавляем результаты тестирования если доступны
if 'test_suite' in locals():
    summary['Тесты пройдено'] = f"{test_suite.passed}/{test_suite.total_tests}"
    summary['Тесты провалено'] = test_suite.failed
    summary['Предупреждения'] = test_suite.warnings

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Значение']
display(summary_df)

print("\n✅ Комплексное тестирование завершено!")
print("\n📝 Следующие шаги:")
print("  1. Проверьте результаты всех разделов")
print("  2. Исправьте найденные проблемы")
print("  3. При необходимости перезапустите модель")
print("  4. Экспортируйте результаты для отчетности")


## 1️⃣2️⃣ Тестирование ковенант (Covenants) <a id="section-12"></a>

**Важно**: Этот раздел должен выполняться после построения модели и валидации результатов.


In [ ]:
display(Markdown("### Расчет и анализ ковенант"))

# Загружаем конфигурацию ковенант
covenant_policy = {}
covenant_cfg = proj_yaml.get('covenants', {})
if covenant_cfg and covenant_cfg.get('enabled', True):
    thresholds = covenant_cfg.get('thresholds', {})
    covenant_policy = {
        "max_leverage_nd_ebitda": thresholds.get("net_debt_to_ebitda_max", 4.0),
        "interest_coverage_min": thresholds.get("interest_coverage_min", 2.0),
        "dscr_min": thresholds.get("dscr_min", 1.2),
        "debt_to_equity_max": thresholds.get("debt_to_equity_max", 1.0),
        "ffo_to_debt_min": thresholds.get("ffo_to_debt_min", 0.15),
        "debt_to_ffo_max": thresholds.get("debt_to_ffo_max", 6.0),
    }
else:
    covenant_policy = {"max_leverage_nd_ebitda": 4.0}

print(f"📊 Пороги ковенант:")
for key, value in covenant_policy.items():
    print(f"  - {key}: {value}")

# Запускаем расчет ковенант (требует построенной модели)
if MODEL_VERSION:
    try:
        covenants_df = run_covenants(ROOT, COMPANY, version=MODEL_VERSION, policy=covenant_policy)
        
        if not covenants_df.empty:
            display(Markdown("#### Результаты ковенант"))
            display(covenants_df)
            
            # Визуализация ковенант
            covenant_metrics = ['NetDebt/EBITDA', 'Interest_Coverage_Ratio', 'DSCR', 'Debt/Equity', 'FFO/NetDebt', 'Debt/FFO']
            
            fig, axes = plt.subplots(2, 3, figsize=(18, 10))
            axes = axes.flatten()
            
            for idx, metric in enumerate(covenant_metrics):
                if metric in covenants_df.columns:
                    ax = axes[idx]
                    years = covenants_df['year'].values
                    values = covenants_df[metric].values
                    
                    ax.plot(years, values, 'o-', label=metric)
                    
                    # Добавляем порог
                    if metric == 'NetDebt/EBITDA' and 'max_leverage_nd_ebitda' in covenant_policy:
                        ax.axhline(y=covenant_policy['max_leverage_nd_ebitda'], color='red', linestyle='--', label='Threshold')
                    elif metric == 'Interest_Coverage_Ratio' and 'interest_coverage_min' in covenant_policy:
                        ax.axhline(y=covenant_policy['interest_coverage_min'], color='red', linestyle='--', label='Threshold')
                    elif metric == 'DSCR' and 'dscr_min' in covenant_policy:
                        ax.axhline(y=covenant_policy['dscr_min'], color='red', linestyle='--', label='Threshold')
                    
                    ax.set_xlabel('Year')
                    ax.set_ylabel(metric)
                    ax.set_title(metric)
                    ax.legend()
                    ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Анализ нарушений
            breach_columns = [c for c in covenants_df.columns if c.startswith('Breach_')]
            if breach_columns:
                display(Markdown("#### Анализ нарушений ковенант"))
                
                breach_summary = []
                for year in covenants_df['year'].values:
                    year_data = covenants_df[covenants_df['year'] == year].iloc[0]
                    breaches = [col.replace('Breach_', '') for col in breach_columns if year_data.get(col, False)]
                    breach_summary.append({
                        'Year': year,
                        'Any_Breach': year_data.get('Any_Breach', False),
                        'Breaches': ', '.join(breaches) if breaches else 'None'
                    })
                
                breach_df = pd.DataFrame(breach_summary)
                display(breach_df)
                
                total_breaches = breach_df['Any_Breach'].sum()
                print(f"\\n📊 Статистика нарушений:")
                print(f"  - Всего лет с нарушениями: {total_breaches} из {len(breach_df)}")
        else:
            print("⚠️ Ковенанты не рассчитаны")
            
    except Exception as e:
        print(f"⚠️ Ошибка расчета ковенант: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Модель еще не построена. Запустите раздел 'Построение модели' сначала.")
